# Diffusion in Solids

In [1]:
from majordome.constants import GAS_CONSTANT
from majordome import diffusion as md
import numpy as np

## Carburizing of steel

In [2]:
def pre_exponential(x, _T):
    return 4.84e-05 * np.exp(-320 * x[0] / GAS_CONSTANT) / (1 - 5 * x[0])

def activation_energy(x, _T):
    return 155_000 - 570_000 * x[0]

Ac = md.PreExponentialFactor(pre_exponential)
Ec = md.ActivationEnergy(activation_energy)

In [3]:
D1 = md.ArrheniusModifiedDiffusivity(
    pre_exponential_factor = Ac,
    activation_energy      = Ec
)
D2 = md.ArrheniusModifiedDiffusivity(
    pre_exponential_func   = pre_exponential,
    activation_energy_func = activation_energy,
)
D3 = md.ArrheniusModifiedDiffusivity(
    pre_exponential_factor = Ac,
    activation_energy_func = activation_energy,
)
D4 = md.ArrheniusModifiedDiffusivity(
    pre_exponential_func   = pre_exponential,
    activation_energy      = Ec,
)

In [4]:
for i, Di in enumerate([D1, D2, D3, D4], start=1):
    print(f"D{i}: {Di([0.01], 1000):.6e} m^2/s")

D1: 5.514355e-13 m^2/s
D2: 5.514355e-13 m^2/s
D3: 5.514355e-13 m^2/s
D4: 5.514355e-13 m^2/s


## Carbonitriding of steel

In [5]:
def f(x):
    return x[0] + 0.72 * x[1]

def A(x):
    return np.exp(-320 * f(x) / GAS_CONSTANT) / (1 - 5 * sum(x))

def E(x):
    return 570_000 * f(x)

Dc = md.ArrheniusModifiedDiffusivity(
    pre_exponential_func   = lambda x, _: 4.84e-05 * (1 - 5 * x[1]) * A(x),
    activation_energy_func = lambda x, _: 155_000 - E(x),
)
Dn = md.ArrheniusModifiedDiffusivity(
    pre_exponential_func   = lambda x, _: 9.10e-05 * (1 - 5 * x[0]) * A(x),
    activation_energy_func = lambda x, _: 168_600 - E(x),
)

Dc_rust = md.slycke.create_carbon_diffusivity()
Dn_rust = md.slycke.create_nitrogen_diffusivity()

In [6]:
Y = [0.01, 0.01]
m = [12.01, 14.01]

M = 1 / ((1-sum(Y)) / 55.85 + sum(y / mi for y, mi in zip(Y, m)))
X = [y * M / mi for y, mi in zip(Y, m)]

T = 1173

print(f"D_C: {Dc(X, T):.6e} | {Dc_rust(X, T):.6e} m²/s")
print(f"D_N: {Dn(X, T):.6e} | {Dn_rust(X, T):.6e} m²/s")

D_C: 3.384958e-11 | 3.384958e-11 m²/s
D_N: 1.517713e-11 | 1.517713e-11 m²/s
